# 1. Import Libraries

In [1]:
from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM
from transformers import pipeline
import os

!pip install deep_translator
from deep_translator import GoogleTranslator

/home/lenovo/Documents/School/Sem6/PBA/pba-task-group/.env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


# 2. Get Model

In [2]:
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 7162.93it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


# 3. Create Prompt Function

In [3]:
def prompt_llm(prompt: str):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.9,
        top_p=0.95,
        repetition_penalty=1.5,
        no_repeat_ngram_size=4
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )


## 3.1 Test Prompt

In [4]:

prompt = """
Paraphrase the following sentence without changing its positive sentiment:
"Film ini sangat menarik dan akting para pemainnya luar biasa."
"""

prompt_llm(prompt)

"It's a well made, highly rated film."

# 4. Count Dataset Sentiments

In [5]:
import pandas as pd

df = pd.read_csv("../../dataset/dataset.csv")

sentiment_counts = {}

for index, row in df.iterrows():
    sentiment = row["manual sentiment"]
    
    if (sentiment not in sentiment_counts):
        sentiment_counts[sentiment] = 1
    else:
        sentiment_counts[sentiment] += 1

highest_sentiment_tuple = ("", 0)

for key in sentiment_counts.keys():
    value = sentiment_counts[key]
    if (value > highest_sentiment_tuple[1]):
        highest_sentiment_tuple = (key, value)

sentiment_generation_counts = {}

for key in sentiment_counts.keys():
    value = sentiment_counts[key]
    _, highest_sentiment_count = highest_sentiment_tuple

    sentiment_generation_counts[key] = highest_sentiment_count -  value

print("The LLM needs to generate more data according to these counts: ")
print(sentiment_generation_counts)

The LLM needs to generate more data according to these counts: 
{'Positive': 348, 'Negative': 0, 'Neutral': 280}


# 5. Generate Prompts

In [6]:
def create_prompt(sentiment: str) -> str:

    examples = {
        "Positive": """
Example (structure only, do NOT reuse wording):
- Talks about optimism in Indonesian exports
- Mentions analysts and business groups
- Ends with positive trade outlook
""",

        "Negative": """
Example (structure only, do NOT reuse wording):
- Describes pressure on export industries
- Includes economist warnings
- Mentions concern about uncertainty
""",

        "Neutral": """
Example (structure only, do NOT reuse wording):
- Announces policy change
- Notes government review in Indonesia
- States uncertainty about long-term impact
"""
    }

    prompt = f"""
Write a {sentiment.lower()} economic news article in English about Donald Trump's tariff policies and Indonesia.

STRICT RULES:
- Do NOT reuse or rephrase any sentence from the example
- Do NOT start with a sentence about "Donald Trump announced" or similar phrasing
- Each sentence must be newly written
- Only use the example for idea structure, not wording
- Write exactly 3 short sentences
- Formal news style

Guidance structure:
{examples[sentiment]}

Output:
Article:
"""
    return prompt

prompts_per_sentiment = {}

for key in sentiment_counts.keys():
    prompts_per_sentiment[key] = create_prompt(key)

prompts_per_sentiment

{'Positive': '\nWrite a positive economic news article in English about Donald Trump\'s tariff policies and Indonesia.\n\nSTRICT RULES:\n- Do NOT reuse or rephrase any sentence from the example\n- Do NOT start with a sentence about "Donald Trump announced" or similar phrasing\n- Each sentence must be newly written\n- Only use the example for idea structure, not wording\n- Write exactly 3 short sentences\n- Formal news style\n\nGuidance structure:\n\nExample (structure only, do NOT reuse wording):\n- Talks about optimism in Indonesian exports\n- Mentions analysts and business groups\n- Ends with positive trade outlook\n\n\nOutput:\nArticle:\n',
 'Negative': '\nWrite a negative economic news article in English about Donald Trump\'s tariff policies and Indonesia.\n\nSTRICT RULES:\n- Do NOT reuse or rephrase any sentence from the example\n- Do NOT start with a sentence about "Donald Trump announced" or similar phrasing\n- Each sentence must be newly written\n- Only use the example for idea

# 6. Create Translation Pipeline

In [7]:
def translate_to_indonesian(text: str):
    return GoogleTranslator(
        source="en",
        target="id"
    ).translate(text)

## 6.1 Test Run

In [8]:

for key in prompts_per_sentiment.keys():
    result = prompt_llm(prompts_per_sentiment[key])
    translated = translate_to_indonesian(result)
    print(key, ": ", result)
    print("Translated: " + translated)

Positive :  President Trump's tariff policy and his presidency are both positive factors that make Indonesia a more attractive target for foreign investment. But in some cases, the broader issue has been the political one. President Donald Trump announced his new policy on Tuesday, and he said that there's a clear resurgence in economic activity in the country and that it's all about the economy.
Translated: Kebijakan tarif Presiden Trump dan masa kepresidenannya merupakan faktor positif yang membuat Indonesia menjadi target investasi asing yang lebih menarik. Namun dalam beberapa kasus, isu yang lebih luas adalah isu politik. Presiden Donald Trump mengumumkan kebijakan barunya pada hari Selasa, dan dia mengatakan bahwa ada peningkatan yang jelas dalam aktivitas ekonomi di negara tersebut dan ini semua tentang perekonomian.
Negative :  The Trump administration announced Wednesday a new tax cut for Indonesia on Thursday, citing the state's economic and financial crisis. At least 11 mill

# 7. Generate New Data into Dataset

In [14]:
file_path = "../../outputs/RAG/rag_results.csv"

if os.path.exists(file_path):
    generated_df = pd.read_csv(file_path)
else:
    generated_df = pd.DataFrame(columns=["sentiment", "text_en", "text_id", "llm_generated"])

existing_counts = generated_df["sentiment"].value_counts().to_dict()

for sentiment in sentiment_generation_counts.keys():
    minus = 0
    if sentiment in existing_counts:
        minus = existing_counts[sentiment]
    amount = sentiment_generation_counts[sentiment] - minus

    for i in range(amount):
        while True:
            try:
                result = prompt_llm(prompts_per_sentiment[sentiment])
                translated = translate_to_indonesian(result)
            
                generated_df.loc[len(generated_df)] = {
                "sentiment": sentiment,
                "text_en": result,
                "text_id": translated,
                "llm_generated": True
            }

                print(sentiment, i + 1, "of", amount)
                print("  Result:", result)
                print("  Translated: ", translated)

                generated_df.to_csv(file_path, index=False)
                break
            except Exception as e:
                print(i, e)

generated_df

Neutral 1 of 280
  Result: The President of the United States announced his tariff policy on Chinese imports and exports, but reaffirmed that it was not based on a reasonable economic assessment of the Chinese product.
  Translated:  Presiden Amerika Serikat mengumumkan kebijakan tarifnya terhadap impor dan ekspor Tiongkok, namun menegaskan kembali bahwa kebijakan tersebut tidak didasarkan pada penilaian ekonomi yang masuk akal terhadap produk Tiongkok.
Neutral 2 of 280
  Result: Donald Trump announced a new tariff on Indonesia -- and the world's most expensive export product -- when he met with Indian officials. The price of the product fell by 2 percent to 1.3 percent, up 6 percent from June, or just 0.25 percent at December's end. "We'll be using that price change in the next few months to raise prices in Indonesia," said Ivan Kaczak, the president's chief economist. That will increase his country's value in the market, he said. Trump has suggested that "the American dollar would ha

,sentiment,text_en,text_id,llm_generated
0,Positive,"Donald Trump's tariff policies, despite the st...","Kebijakan tarif Donald Trump, meskipun pasarny...",True
1,Positive,Donald Trump's tariff policy has created optim...,Kebijakan tarif Donald Trump telah menciptakan...,True
2,Positive,Donald Trump's tariff policy creates optimism ...,Kebijakan tarif Donald Trump menciptakan optim...,True
3,Positive,Donald Trump's tariff policy on Indonesia will...,Kebijakan tarif Donald Trump terhadap Indonesi...,True
4,Positive,The US has introduced a tariff on the use of m...,Amerika telah menerapkan tarif terhadap penggu...,True
...,...,...,...,...
623,Neutral,Donald Trump made tough comments on a proposed...,Donald Trump melontarkan komentar keras mengen...,True
624,Neutral,Donald Trump's tariff policy was a major econo...,"Namun, kebijakan tarif Donald Trump merupakan ...",True
625,Neutral,"A look at Donald Trump's tariff policy, includ...","Sekilas tentang kebijakan tarif Donald Trump, ...",True
626,Neutral,President Donald Trump's tariff policy may lea...,Kebijakan tarif Presiden Donald Trump dapat me...,True
